# Projekt 01 (basic): Ein MLP von Hand — Backpropagation in purem NumPy

**Ziel:** Du implementierst ein Multilayer-Perzeptron komplett selbst — Forward Pass, Backpropagation (die $\delta$-Rekursion aus dem Skript, Abschnitt 1.5), Gradient Checking und Training mit Minibatch-SGD + Momentum. Kein PyTorch, kein sklearn-Modell: nur NumPy. Danach weißt du *exakt*, was `loss.backward()` tut.

**Daten:** `make_moons` — zwei ineinandergreifende Halbmonde. Synthetisch und bewusst gewählt: nicht linear separierbar (ein lineares Modell scheitert beweisbar), aber 2D (wir können die gelernte Entscheidungsgrenze *sehen*), reproduzierbar per Seed.

Mathematische Referenz — die vier Backprop-Gleichungen, die wir implementieren:

$$\boldsymbol{\delta}^{(L)} = \mathbf{p} - \mathbf{y} \qquad \text{(Sigmoid + BCE)}$$
$$\boldsymbol{\delta}^{(\ell)} = \big({W^{(\ell+1)}}^\top \boldsymbol{\delta}^{(\ell+1)}\big) \odot \sigma'\big(\mathbf{z}^{(\ell)}\big)$$
$$\nabla_{W^{(\ell)}} \mathcal{L} = \tfrac{1}{B}\, \boldsymbol{\delta}^{(\ell)} {\mathbf{A}^{(\ell-1)}}^\top \qquad \nabla_{\mathbf{b}^{(\ell)}} \mathcal{L} = \tfrac{1}{B} \textstyle\sum_{\text{Batch}} \boldsymbol{\delta}^{(\ell)}$$

(Batch-Konvention hier: Spalten = Beispiele, d. h. $\mathbf{A}^{(\ell)} \in \mathbb{R}^{d_\ell \times B}$.)

> **Arbeitsweise:** Fülle die `TODO`-Stellen. Nach jedem Schritt gibt es eine Prüfzelle (Gradient Check bzw. Accuracy), die dir sagt, ob dein Code stimmt. Die Musterlösung liegt in `solution/solution.ipynb` — erst selbst probieren!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)

# Daten: 2 Halbmonde, bewusst verrauscht (noise=0.25), damit die Aufgabe nicht trivial ist
X, y = make_moons(n_samples=1000, noise=0.25, random_state=42)
X = (X - X.mean(axis=0)) / X.std(axis=0)   # standardisieren (wichtig fuer stabile Gradienten)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=12, alpha=0.7)
ax.set_title("make_moons (standardisiert, Trainingsdaten)")
plt.show()
print(X_train.shape, X_test.shape)

## Schritt 1: Architektur und Forward Pass

Netz: $2 \to 32 \to 32 \to 1$, verborgene Schichten mit **ReLU**, Ausgabe ein einzelnes **Logit** $z$ (die Sigmoid stecken wir aus Stabilitätsgründen in die Loss).

**Binäre Cross-Entropy mit Logits**, numerisch stabil (vgl. Skript 1.3 — nie Sigmoid und Log getrennt rechnen):

$$\mathcal{L}(z, y) = \max(z, 0) - yz + \log\big(1 + e^{-\lvert z \rvert}\big)$$

*(Nachrechnen lohnt sich: Für $z \ge 0$ und $z < 0$ ist das jeweils exakt $-[y\log\sigma(z) + (1-y)\log(1-\sigma(z))]$, aber ohne Overflow.)*

**Initialisierung:** He ($\sigma_W^2 = 2/d_{\text{in}}$), da ReLU — Herleitung im Skript 2.2.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)

def relu_prime(z):
    return (z > 0).astype(z.dtype)

def sigmoid(z):
    # stabil: fuer z>=0 via 1/(1+e^-z), fuer z<0 via e^z/(1+e^z)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def bce_with_logits(z, y):
    # z, y: shape (1, B); Rueckgabe: mittlerer Verlust (Skalar)
    return np.mean(np.maximum(z, 0) - y * z + np.log1p(np.exp(-np.abs(z))))

class MLP:
    """MLP mit ReLU-Hidden-Layern und 1 Logit-Ausgang. Spalten = Beispiele."""

    def __init__(self, layer_sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = layer_sizes
        self.W, self.b = [], []
        for d_in, d_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.W.append(rng.normal(0.0, np.sqrt(2.0 / d_in), size=(d_out, d_in)))
            self.b.append(np.zeros((d_out, 1)))

    def forward(self, A0):
        """A0: (d_0, B). Speichert Zs und As fuer den Backward Pass."""
        self.Zs, self.As = [], [A0]
        A = A0
        L = len(self.W)
        for l in range(L):
            Z = self.W[l] @ A + self.b[l]
            self.Zs.append(Z)
            A = relu(Z) if l < L - 1 else Z   # letzte Schicht: rohes Logit
            self.As.append(A)
        return A   # (1, B) Logits

    def backward(self, y):
        """y: (1, B) mit Werten in {0,1}. Fuellt self.dW, self.db (gemittelt ueber den Batch)."""
        B = y.shape[1]
        L = len(self.W)
        self.dW = [None] * L
        self.db = [None] * L

        # TODO 1: Fehlersignal der Ausgabeschicht.
        # Fuer Sigmoid+BCE gilt delta^(L) = p - y (Skript 1.3/1.5). p = sigmoid(Logits).
        delta = ...                                               # (1, B)

        for l in range(L - 1, -1, -1):
            # TODO 2: Parametergradienten als aeusseres Produkt "Fehler mal Eingang",
            # ueber den Batch gemittelt: dW = delta @ A^(l-1)^T / B ; db = Zeilenmittel von delta.
            self.dW[l] = ...
            self.db[l] = ...
            if l > 0:
                # TODO 3: Backprop-Rekursion: delta^(l-1) = (W^(l)^T delta^(l)) * relu'(Z^(l-1))
                delta = ...

    def loss(self, X, y):
        return bce_with_logits(self.forward(X), y)

    def predict(self, X):
        return (self.forward(X) > 0).astype(int)   # Logit > 0  <=>  p > 0.5

## Schritt 2: Gradient Checking — traue keinem selbstgeschriebenen Backprop

Wir vergleichen den analytischen Gradienten mit der **zentralen Differenz** (Fehler $O(\epsilon^2)$, Skript 1.6):

$$\frac{\partial \mathcal{L}}{\partial \theta_j} \approx \frac{\mathcal{L}(\theta + \epsilon \mathbf{e}_j) - \mathcal{L}(\theta - \epsilon \mathbf{e}_j)}{2\epsilon}$$

und messen die relative Abweichung $\dfrac{\lVert g_{\text{ana}} - g_{\text{num}} \rVert}{\lVert g_{\text{ana}} \rVert + \lVert g_{\text{num}} \rVert}$. In float64 mit $\epsilon = 10^{-5}$ erwarten wir $\lesssim 10^{-7}$.

Ein subtiler Punkt: ReLU ist bei $z = 0$ nicht differenzierbar — liegt ein $z$ zufällig *sehr* nah an 0, kann die zentrale Differenz über den Knick springen und der Check schlägt scheinbar fehl. Bei zufälligen Daten passiert das praktisch nie; wir prüfen zusätzlich auf einem kleinen Netz, wo alles überschaubar ist.

In [ ]:
def gradient_check(model, X, y, eps=1e-5, n_checks=60, seed=0):
    """Vergleicht Backprop-Gradienten mit zentralen Differenzen an zufaelligen Koordinaten."""
    rng = np.random.default_rng(seed)
    model.forward(X)
    model.backward(y)
    ana_all, num_all = [], []
    params = [(W, dW) for W, dW in zip(model.W, model.dW)] + \
             [(b, db) for b, db in zip(model.b, model.db)]
    for theta, grad in params:
        flat_theta = theta.ravel()
        flat_grad = grad.ravel()
        for idx in rng.choice(flat_theta.size, size=min(n_checks, flat_theta.size), replace=False):
            orig = flat_theta[idx]
            flat_theta[idx] = orig + eps
            L_plus = model.loss(X, y)
            flat_theta[idx] = orig - eps
            L_minus = model.loss(X, y)
            flat_theta[idx] = orig
            num_all.append((L_plus - L_minus) / (2 * eps))
            ana_all.append(flat_grad[idx])
    ana, num = np.array(ana_all), np.array(num_all)
    rel = np.linalg.norm(ana - num) / (np.linalg.norm(ana) + np.linalg.norm(num))
    return rel

# Check auf einem kleinen Netz in float64
Xc = rng.normal(size=(2, 20))            # (d0, B) — Spalten = Beispiele!
yc = rng.integers(0, 2, size=(1, 20)).astype(float)
small = MLP([2, 5, 4, 1], seed=1)
rel_err = gradient_check(small, Xc, yc)
print(f"relative Abweichung: {rel_err:.2e}")
assert rel_err < 1e-7, "Backprop fehlerhaft — pruefe deine TODOs 1-3!"
print("Gradient Check bestanden — der Backprop-Code ist korrekt.")

## Schritt 3: Training mit Minibatch-SGD + Momentum

Update-Regeln (Skript 2.1): $\mathbf{v} \leftarrow \mu \mathbf{v} + \mathbf{g}$, $\;\theta \leftarrow \theta - \eta\, \mathbf{v}$ mit $\mu = 0.9$.

Wir loggen Trainings- und Testverlust pro Epoche — die Kurven sind das wichtigste Diagnoseinstrument (Overfitting? Lernrate zu groß/klein?).

In [ ]:
def train(model, X_tr, y_tr, X_te, y_te, epochs=200, batch_size=64, lr=0.1, momentum=0.9, seed=0):
    rng = np.random.default_rng(seed)
    Xtr, ytr = X_tr.T, y_tr.reshape(1, -1).astype(float)   # Spalten = Beispiele
    Xte, yte = X_te.T, y_te.reshape(1, -1).astype(float)
    n = Xtr.shape[1]
    vW = [np.zeros_like(W) for W in model.W]
    vb = [np.zeros_like(b) for b in model.b]
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            Xb, yb = Xtr[:, idx], ytr[:, idx]
            model.forward(Xb)
            model.backward(yb)
            for l in range(len(model.W)):
                # TODO 4: Momentum-Update (Skript 2.1):
                #   v <- momentum * v + gradient ;  theta <- theta - lr * v
                # ... fuer W[l] (mit vW[l], dW[l]) und b[l] (mit vb[l], db[l])
                raise NotImplementedError
        hist["train_loss"].append(model.loss(Xtr, ytr))
        hist["test_loss"].append(model.loss(Xte, yte))
        hist["test_acc"].append((model.predict(Xte) == yte).mean())
    return hist

model = MLP([2, 32, 32, 1], seed=3)
hist = train(model, X_train, y_train, X_test, y_test)
print(f"Test-Accuracy nach Training: {hist['test_acc'][-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist["train_loss"], label="Train")
axes[0].plot(hist["test_loss"], label="Test")
axes[0].set_xlabel("Epoche"); axes[0].set_ylabel("BCE-Loss"); axes[0].legend(); axes[0].set_title("Lernkurven")
axes[1].plot(hist["test_acc"])
axes[1].set_xlabel("Epoche"); axes[1].set_ylabel("Test-Accuracy"); axes[1].set_title("Accuracy")
plt.tight_layout(); plt.show()

## Schritt 4: Die gelernte Entscheidungsgrenze

Der eigentliche Zweck der 2D-Daten: Wir können *sehen*, welche Funktion das Netz gelernt hat. Ein lineares Modell könnte hier nur eine Gerade ziehen — das MLP hat aus den ReLU-Knicken eine gekrümmte Grenze komponiert (stückweise linear, wenn man genau hinsieht!).

In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()].T                    # (2, 90000)
    probs = sigmoid(model.forward(grid)).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6, 5))
    cs = ax.contourf(xx, yy, probs, levels=20, cmap="coolwarm", alpha=0.75)
    ax.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=15, linewidths=0.3)
    fig.colorbar(cs, label="P(Klasse 1)")
    ax.set_title(title)
    plt.show()

plot_decision_boundary(model, X_test, y_test, "Gelernte Entscheidungsgrenze (Testdaten)")

## Schritt 5: Experiment — Kapazität und Entscheidungsgrenze

Wie verändert die Netzbreite die gelernte Funktion? Wir trainieren drei Netze: unterparametrisiert ($2{\to}2{\to}1$), moderat ($2{\to}16{\to}1$), breit ($2{\to}128{\to}128{\to}1$), und vergleichen Grenze + Test-Accuracy. Achte darauf, wie das 2-Neuronen-Netz strukturell scheitert (es hat nur 2 ReLU-Knicke zur Verfügung) und ob das breite Netz overfittet (Skript 3.6: nicht unbedingt!).

In [ ]:
for sizes in [[2, 2, 1], [2, 16, 1], [2, 128, 128, 1]]:
    m = MLP(sizes, seed=3)
    h = train(m, X_train, y_train, X_test, y_test)
    plot_decision_boundary(m, X_test, y_test,
                           f"Architektur {sizes} — Test-Acc {h['test_acc'][-1]:.3f}")

## Fazit

- Die vier Backprop-Gleichungen aus dem Skript sind der *komplette* Kern des Deep Learning — alles Weitere (Adam, BatchNorm, CNNs) sind Verfeinerungen darum herum.
- **Gradient Checking** ist die Standardtechnik, um handgeschriebene Gradienten zu verifizieren — relative Abweichung $< 10^{-7}$ in float64 oder es ist ein Bug.
- Kapazität formt die Entscheidungsgrenze: zu klein → strukturelles Underfitting; groß → oft trotzdem gute Generalisierung (Double Descent, Skript 3.6).

**Weiter geht's in Projekt 02:** dasselbe Prinzip, aber mit PyTorch, echten Bilddaten (Fashion-MNIST), CNNs und einem systematischen Regularisierungs-/Optimierer-Vergleich.